In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# 1. Load and clean your data (using the interactive input from earlier)
df = pd.read_csv("/kyr_querries.csv", sep=',', on_bad_lines='skip')
df = df.drop_duplicates(subset="query")

# 2. Split into train and test sets (e.g., 80% training, 20% testing)
train, test = train_test_split(df, test_size=0.2, random_state=42)

# 3. Vectorize the queries
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2)
)
X_train = vectorizer.fit_transform(train["query"])
X_test = vectorizer.transform(test["query"])

# 4. Train the model and make predictions
model = LogisticRegression(
    max_iter=1000
)
model.fit(
    X_train,
    train["intent"]
)
predictions = model.predict(X_test)


from sklearn.metrics import classification_report

print(
    classification_report(
        test["intent"],
        predictions
    )
)

import os
import joblib

# Build an intent -> domain lookup from the training data
# (assumes each intent maps to exactly one domain, which your taxonomy implies)
intent_to_domain = df.groupby("intent")["domain"].first().to_dict()

# Save everything the RAG pipeline will need
os.makedirs("ps25/model/data/classifier", exist_ok=True)

joblib.dump(vectorizer, "ps25/model/data/classifier/tfidf_vectorizer.joblib")
joblib.dump(model, "ps25/model/data/classifier/intent_classifier.joblib")
joblib.dump(intent_to_domain, "ps25/model/data/classifier/intent_to_domain.joblib")

print("Saved vectorizer, classifier, and intent-to-domain map.")
print("Sample mapping:", dict(list(intent_to_domain.items())[:5]))

ModuleNotFoundError: No module named 'sklearn'